# Tutorial: LLM Gateway

Audience:
- Developers using the LangChain-based gateway and classifier wrapper.

Prerequisites:
- Optional `OPENAI_API_KEY`; mock provider works without external keys.

Learning goals:
- Initialize an LLM gateway by provider.
- Classify a sample headline directly.
- Use `EventClassifier` cache/audit wrapper.
- Probe provider availability.


## Outline

1. Setup imports and provider selection
2. Classify events directly with gateway
3. Classify with `EventClassifier`
4. Test multiple provider backends


In [ ]:
import os

from swing_screener.intelligence.llm import EventClassifier, get_llm_gateway


## Step 1 - Create gateway based on available credentials

Use OpenAI if key exists; otherwise fall back to `mock` provider.


In [ ]:
api_key = os.environ.get("OPENAI_API_KEY")
provider_name = "openai" if api_key else "mock"

gateway = get_llm_gateway(provider_name=provider_name, model="gpt-4o-mini")
print(f"Gateway created: provider={gateway.provider_name}, model={gateway.model_name}")
print(f"Available: {gateway.is_available()}")


## Step 2 - Direct gateway classification

Classify one sample earnings headline and inspect structured fields.


In [ ]:
if gateway.is_available():
    headline = "NVDA reports record earnings, revenue up 25% year-over-year"
    result = gateway.classify_event(headline=headline, snippet="")

    print("Classification result:")
    print(f"  Event type: {result.event_type.value}")
    print(f"  Severity: {result.severity.value}")
    print(f"  Confidence: {result.confidence}")
    print(f"  Is material: {result.is_material}")
    print(f"  Primary symbol: {result.primary_symbol}")
    print(f"  Secondary symbols: {result.secondary_symbols}")
    print(f"  Summary: {result.summary}")
else:
    print(f"Gateway not available: {gateway.availability_error}")


## Step 3 - Wrap with `EventClassifier` for caching and audit

This wrapper stores repeat classifications and writes audit artifacts.


In [ ]:
classifier = EventClassifier(
    provider=gateway,
    cache_path="data/intelligence/llm_cache.json",
    audit_path="data/intelligence/llm_audit",
    enable_cache=True,
    enable_audit=True,
)

headline = "AAPL announces new product launch, analysts upgrade to buy"
snippet = "Apple unveiled its latest MacBook Pro with M4 chip."

result = classifier.classify(
    headline=headline,
    snippet=snippet,
    source="news_api",
    timestamp="2024-01-15T10:30:00Z",
)

print("Classification result:")
print(f"  Event type: {result.classification.event_type.value}")
print(f"  Severity: {result.classification.severity.value}")
print(f"  Confidence: {result.classification.confidence}")
print(f"  Summary: {result.classification.summary}")


## Step 4 - Provider availability checks

Quick diagnostic loop for configured provider backends.


In [ ]:
test_providers = [
    ("mock", "mock-classifier"),
    ("ollama", "mistral:7b-instruct"),
]

for prov, model in test_providers:
    try:
        gw = get_llm_gateway(provider_name=prov, model=model)
        print(f"{prov}: available={gw.is_available()}")
    except Exception as e:
        print(f"{prov}: error={e}")
